In [1]:
# Parameters
UF = "SP"


## Seção 4.4.2 - Ferramenta de visualização de ultrapassagens dos padrões de qualidade do ar no Brasil

Este notebook reproduz a figura 37 da seção 4.4.2 do Relatório Anual de Qualidade do Ar

### Figura 37 - Ferramenta de visualização de ultrapassagensências dos padrões de qualidade do ar no Brasil

Foi desenvolvido um código em Python capaz de processar os dados das estações de monitoramento e disponibilizá-los em uma interface interativa. 

A ferramenta foi concebida para permitir a consulta e a visualização das estatísticas de ultrapassagens aos padrões de qualidade do ar estabelecidos pela Resolução Conama nº 506/2024, possibilitando a exploração dos resultados por estação, poluente e período de análise. 

A ferramenta também disponibiliza a opção de <i>download</i> dos dados filtrados em formato .csv, permitindo o uso dos resultados em análises complementares, relatórios técnicos ou estudos regionais.

### Observação:

Diferente das ferramentas anteriores (Figuras 34, 35 e 36), esta não depende de nenhum arquivo intermediário gerado localmente — basta executar a célula abaixo, e todo o carregamento de dados acontece em tempo real, no navegador, diretamente a partir dos links (o código acessa eles automaticamente):

https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages

https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv

https://arquivos.lcqar.ufsc.br/data/databases/stations/fases_CONAMA506.csv

In [2]:
import json
import requests
from IPython.display import IFrame
import base64

# Mapeamento sigla -> código IBGE (necessário para a API de malhas)
UF_CODES = {
    "AC": 12, "AL": 27, "AP": 16, "AM": 13, "BA": 29, "CE": 23, "DF": 53,
    "ES": 32, "GO": 52, "MA": 21, "MT": 51, "MS": 50, "MG": 31, "PA": 15,
    "PB": 25, "PR": 41, "PE": 26, "PI": 22, "RJ": 33, "RN": 24, "RS": 43,
    "RO": 11, "RR": 14, "SC": 42, "SP": 35, "SE": 28, "TO": 17,
}


def build_stations_explorer(UF=None):
    # -------------------------------------------------------------------
    # Fronteira do estado (opcional, apenas se UF for informado)
    # -------------------------------------------------------------------
    gj_boundary = "null"
    if UF:
        uf_code = UF_CODES.get(UF.upper())
        if uf_code is None:
            raise ValueError(f"UF '{UF}' não reconhecida.")
        boundary_url = (
            f"https://servicodados.ibge.gov.br/api/v2/malhas/{uf_code}"
            f"?resolucao=2&formato=application/vnd.geo+json"
        )
        resp = requests.get(boundary_url)
        if resp.ok:
            gj_boundary = resp.text

    uf_filter_js = json.dumps(UF.upper()) if UF else "null"

    # Ao contrário das Figuras 34/35/36, esta ferramenta não depende de nenhum arquivo
    # intermediário gerado localmente: todo o carregamento de dados (metadados, fases
    # CONAMA 506/2024 e séries por estação/poluente) é feito em tempo real, no navegador,
    # via Papa.parse() diretamente das URLs remotas listadas na célula "Obs" acima.
    html_code = fr"""<!-- Estilo -->
<style>
/* Reestilizando botões */
.appBtn {{
  appearance: none;
  background-color: #FAFBFC;
  border: 1px solid rgba(27, 31, 35, 0.15);
  border-radius: 6px;
  box-shadow: rgba(27, 31, 35, 0.04) 0 1px 0, rgba(255, 255, 255, 0.25) 0 1px 0 inset;
  box-sizing: border-box;
  color: #24292E;
  cursor: pointer;
  display: inline-block;
  font-family: -apple-system, system-ui, "Segoe UI", Helvetica, Arial, sans-serif;
  font-size: 14px;
  font-weight: 500;
  line-height: 20px;
  padding: 6px 16px;
  transition: background-color 0.2s;
  vertical-align: middle;
  background: unset;
}}
.appBtn:hover {{ background-color: #F3F4F6; }}
.appBtn:disabled {{ color: #959DA5; cursor: default; }}
    
/* Tabela de estatísticas */
.stats-table {{
  border-collapse: collapse;
  width: 100%;
  margin: 15px auto; /* centraliza horizontalmente */
  max-width: 500px;
  margin-top: 20px;
  font-family: Arial, sans-serif;
  background-color: #FAFBFC;
  border: 1px solid rgba(27,31,35,0.15);
  border-radius: 8px;
  overflow: hidden;
}}
.stats-table th, .stats-table td {{
  padding: 8px 12px;
  text-align: left;
}}
.stats-table th {{
  background-color: #F3F4F6;
  font-weight: 600;
}}
.stats-table tr:nth-child(even) {{
  background-color: #F9F9F9;
}}
    
body {{ font-family: Inter, system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial; margin: 12px; }}
label {{ margin-right:6px; }}
select, input {{ margin-right: 14px; }}
#mapDiv {{ border-radius:6px; box-shadow: 0 1px 4px rgba(0,0,0,0.08); }}
</style>

<!-- Dependências -->
<script src="https://cdn.plot.ly/plotly-2.26.1.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/papaparse@5.4.1/papaparse.min.js"></script>
<script id="gj-boundary-stations" type="application/json">
{gj_boundary}
</script>

<div id="mapDiv" style="width:100%; height:360px; margin-bottom:12px;"></div>

<div>
  <label>Estação:</label>
  <select id="stationSelect"></select> <!-- agora é só display -->

  <label>Poluente:</label>
  <select id="pollutantSelect" ></select>

  <label>Tempo de média:</label>
  <select id="aveTimeSelect" ></select>
  <br>
  <br>
  <label>Início:</label>
  <input type="datetime-local" id="startDate">

  <label>Fim:</label>
  <input type="datetime-local" id="endDate">
  <br>
  <br>
  <button id="loadDataBtn" class="btn appBtn">Carrega dados</button>
  <button id="plotBtn" class="btn appBtn" disabled>Plot</button>
  <button id="downloadBtn" class="btn appBtn" disabled>Download CSV</button>
</div>

<div id="plotDiv" style="width:100%; height:600px;"></div>
<div id="statsTable" style="width:80%; margin-top:20px;"></div>


<script>
/* ========== Config ========== */
const DATA_FOLDER = 'https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages';
const METADATA_CSV = 'https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv';
const FASES_CSV = 'https://arquivos.lcqar.ufsc.br/data/databases/stations/fases_CONAMA506.csv';

/* ========== Filtro de UF (opcional, definido em Python) ========== */
const FILTER_UF = {uf_filter_js};
const boundaryRaw = document.getElementById('gj-boundary-stations').textContent.trim();
const dataBoundary = (boundaryRaw && boundaryRaw !== "null") ? JSON.parse(boundaryRaw) : null;

function stationMatchesUF(idMma){{
  if(!FILTER_UF) return true;
  const prefix = (idMma || '').toString().trim().slice(0,2).toUpperCase();
  return prefix === FILTER_UF;
}}

/* ========== Estado ========== */
let metadata = [];
let fasesData = [];
let stationPollutants = {{}};
let currentData = [];   // dados crus do CSV carregado (array de objetos)
let filteredData = [];  // dados preparados para plotagem (array {{datetime, valor}})
let stationIds = [];    // ids únicos com coordenadas para o mapa
let stationCoordsArr = []; // array de objetos {{ID_MMA, LATITUDE, LONGITUDE, ID_OEMA, ...}}
let mapDivEl;

/* ========== Util: wrapper Papa.parse -> Promise ========== */
function parseCSV(url) {{
  return new Promise((resolve, reject) => {{
    Papa.parse(url, {{
      download: true,
      header: true,
      skipEmptyLines: true,
      complete: res => resolve(res.data || []),
      error: err => reject(err)
    }});
  }});
}}

/* ========== Inicialização: carrega metadados e fases ========== */
Promise.all([parseCSV(METADATA_CSV), parseCSV(FASES_CSV)])
  .then(([metaRows, fasesRows]) => {{
    // filtra linhas úteis
    //metadata = metaRows.filter(r => (r.ID_MMA || '').toString().trim() && (r.POLUENTE || '').toString().trim());
    const commonPollutants = [
      "MP10", "MP25", "CO", "NO2", "SO2", "O3", "Pb"
    ];

    // filtra linhas úteis, mantém apenas poluentes comuns e aplica filtro de UF (se houver)
    metadata = metaRows.filter(r => {{
      const id = (r.ID_MMA || '').toString().trim();
      const pol = (r.POLUENTE || '').toString().trim();
      if(!id || !pol || !commonPollutants.includes(pol)) return false;
      return stationMatchesUF(id);
    }});
    fasesData = fasesRows; // assumimos cabeçalhos corretos em fases CSV

    if(FILTER_UF && !metadata.length){{
      alert(`Nenhuma estação encontrada para UF ${{FILTER_UF}}.`);
    }}

    buildStationPollutantsMap();
    populatePollutantSelect();
    plotStationsMap();
    populateStationsDropdown();
    updateAveTimeDropdown();

    // listeners
    document.getElementById('pollutantSelect').addEventListener('change', () => {{
      // quando mudar poluente fora do mapa, atualiza lista de estações disponíveis (mantemos dropdown de estação livre)
      //populateStationSelectByPol();
      updateAveTimeDropdown();
      document.getElementById('plotBtn').disabled = true;
      document.getElementById('downloadBtn').disabled = true;
    }});

    document.getElementById('loadDataBtn').addEventListener('click', loadStationData);
    document.getElementById('plotBtn').addEventListener('click', doPlot);
    document.getElementById('downloadBtn').addEventListener('click', downloadFilteredCSV);
  }})
  .catch(err => {{
    console.error('Erro carregando CSVs:', err);
    alert('Erro ao carregar metadados ou fases. Veja console.');
  }});

/* ========== Construir mapa auxiliar ========== */
function plotStationsMap() {{
  // criamos um mapa único de estações com coordenadas (ID_MMA -> primeira linha com lat/lon)
  const stationMap = {{}};
  metadata.forEach(r=>{{
    const id = (r.ID_MMA||'').toString().trim();
    const lat = parseFloat((r.LATITUDE||'').toString().replace(',','.'));
    const lon = parseFloat((r.LONGITUDE||'').toString().replace(',','.'));
    if(!id || Number.isNaN(lat) || Number.isNaN(lon)) return;
    if(!stationMap[id]) stationMap[id] = r; // mantém a primeira ocorrência
  }});
  stationCoordsArr = Object.values(stationMap);
  if(!stationCoordsArr.length) return;

  stationIds = stationCoordsArr.map(r => (r.ID_MMA||'').toString().trim());
  const lats = stationCoordsArr.map(r => parseFloat(r.LATITUDE));
  const lons = stationCoordsArr.map(r => parseFloat(r.LONGITUDE));
  const texts = stationCoordsArr.map(r => `${{r.ID_MMA}} — ${{r.ID_OEMA || ''}}`);
  const colors = metadata.map(r => r.BASE_DADOS === 'True' ? 'blue' : '#808080'
  );

  const trace = {{
    type: "scattermapbox",
    lat: lats,
    lon: lons,
    mode: "markers",
    text: texts,
    showlegend: false,
    customdata: stationIds,
    marker: {{ size: 10, color: colors, opacity: Array(stationIds.length).fill(0.8) }},
    hoverinfo: "text"
  }};

    // Adicionando traces fantasmas, só para ter legenda
  const legendBlue = {{
    type: "scattermapbox",
    mode: "markers",
    lat: [null], lon: [null],
    marker: {{ size: 10, color: 'blue' }},
    name: "Com base de dados",
    showlegend: true,
    hoverinfo: "none"
  }};

  const legendGray = {{
    type: "scattermapbox",
    mode: "markers",
    lat: [null], lon: [null],
    marker: {{ size: 10, color: '#808080' }},
    name: "Sem base de dados",
    showlegend: true,
    hoverinfo: "none"
  }};

  // centro/zoom padrão (Brasil) ou ajustado à UF se soubermos as coords médias
  let centerLat = lats[0], centerLon = lons[0], zoom = 2;
  if(FILTER_UF && lats.length){{
    centerLat = lats.reduce((a,b)=>a+b,0)/lats.length;
    centerLon = lons.reduce((a,b)=>a+b,0)/lons.length;
    zoom = 6;
  }}

  const mapboxLayers = [];
  if(dataBoundary){{
    mapboxLayers.push({{
      sourcetype: 'geojson',
      source: dataBoundary,
      type: 'line',
      color: '#000',
      line: {{ width: 2 }},
      below: 'traces'
    }});
  }}

  const layout = {{
    mapbox: {{ style:  'carto-positron', center: {{ lat: centerLat, lon: centerLon }}, zoom: zoom, layers: mapboxLayers }},
    margin: {{ t: 0, b: 0, l: 0, r: 0 }},
    legend: {{
        title: {{ text: 'Situação das Estações' }},
        x: 0.99,             // lado direito
        xanchor: 'right', // ancora à direita
        y: 0.12,             // parte de baixo
        yanchor: 'bottom',// ancora na base
        bgcolor: 'rgba(255,255,255,0.7)', // fundo semitransparente (para ver o mapa)
        bordercolor: 'rgba(0,0,0,0.3)',
        borderwidth: 1,
        font: {{ size: 12 }}
    }}
  }};

  mapDivEl = document.getElementById('mapDiv');
  Plotly.newPlot(mapDivEl, [trace, legendBlue, legendGray], layout, {{ responsive: true }});

  // clique no mapa seleciona estação
  mapDivEl.on('plotly_click', function(data){{
    if(data.points && data.points.length>0) {{
      const stationId = data.points[0].customdata;
      selectStation(stationId);
    }}
  }});
}}

function populateStationsDropdown() {{
  const stationSel = document.getElementById('stationSelect');
  stationSel.innerHTML = '';
  stationCoordsArr.forEach(
    (r) => {{
      const option = document.createElement('option');
      option.value = r.ID_MMA;
      option.textContent = `${{r.ID_MMA}} - ${{r.ID_OEMA}}`;

      stationSel.appendChild(option);
    }});
}}

/* quando clicar no mapa: centraliza, destaca, atualiza dropdowns */
function selectStation(stationId){{
  // atualiza marcador (cores/tamanhos/opacidade)
  // const newColors = stationIds.map(id => id === stationId ? 'lime' : 'blue');
  const newColors = stationCoordsArr.map(r => r.ID_MMA === stationId ? 'lime' : (r.BASE_DADOS === 'True' ? 'blue' : '#808080'));
  // globalThis.metadata = metadata;
  // globalThis.stationId = stationId;
  const newSizes  = stationIds.map(id => id === stationId ? 16 : 10);
  const newOpac   = stationIds.map(id => id === stationId ? 1 : 0.8);
  
  Plotly.restyle(mapDivEl, {{
    'marker.color': [newColors],
    'marker.size': [newSizes],
    'marker.opacity': [newOpac]
  }}, [0]);

  // centraliza mapa na estação
  const idx = stationIds.indexOf(stationId);
  if(idx >= 0 && stationCoordsArr[idx]) {{
    Plotly.relayout(mapDivEl, {{
      'mapbox.center.lat': parseFloat(stationCoordsArr[idx].LATITUDE),
      'mapbox.center.lon': parseFloat(stationCoordsArr[idx].LONGITUDE),
      'mapbox.zoom': 10
    }});
  }}

  // encontra metadado da estação selecionada
  // const meta = metadata.find(m => m.ID_MMA === stationId);

  // // atualiza dropdown
  const stSel = document.getElementById('stationSelect');
  stSel.value = stationId
  // stSel.innerHTML = '';

  // const opt = document.createElement('option');
  // opt.value = stationId;
  // opt.textContent = `${{stationId}} - ${{meta ? meta.ID_OEMA : ''}}`;
  // stSel.appendChild(opt);

  // Desabilita botão "Carrega dados" para dados inexistentes
  const haveData = metadata.some(r =>
    r.ID_MMA === stationId && String(r.BASE_DADOS).toLowerCase() === 'true'
  );

  const loadButton = document.getElementById('loadDataBtn')
  if(!haveData){{
    loadButton.textContent = 'Sem dados para carregar'
    loadButton.disabled = true
  }} else {{
    loadButton.textContent = 'Carregar dados'
    loadButton.disabled = false
  }}


  // atualiza poluentes disponíveis para a estação
  const polSel = document.getElementById('pollutantSelect');
  polSel.innerHTML = '';
  const pols = (stationPollutants[stationId] || []).slice().sort();
  if(pols.length) {{
    pols.forEach(p => {{ const o=document.createElement('option'); o.value=p; o.textContent=p; polSel.appendChild(o); }});
    polSel.selectedIndex = 0;
    updateAveTimeDropdown();
  }} else {{
    // se não houver poluentes listados, mantém as opções globais
    populatePollutantSelect();
  }}

  // desabilita botões até carregar os dados
  document.getElementById('plotBtn').disabled = true;
  document.getElementById('downloadBtn').disabled = true;
}}

/* ========== Mapeamento estação->poluentes ========== */
function buildStationPollutantsMap(){{
  stationPollutants = {{}};
  metadata.forEach(row=>{{
    const id = (row.ID_MMA||'').toString().trim();
    const pol = (row.POLUENTE||'').toString().trim();
    if(!id || !pol) return;
    if(!stationPollutants[id]) stationPollutants[id] = new Set();
    stationPollutants[id].add(pol);
  }});
  for(const st in stationPollutants) stationPollutants[st] = Array.from(stationPollutants[st]);
}}

/* ========== Popula selects (poluentes / estações por poluente) ========== */
function populatePollutantSelect() {{
  const sel = document.getElementById('pollutantSelect');
  sel.innerHTML = '';
  //const polSet = new Set(metadata.map(r => (r.POLUENTE||'').toString().trim()).filter(Boolean));
  //Array.from(polSet).sort().forEach(pol=>{{
  //  const opt = document.createElement('option'); opt.value = pol; opt.textContent = pol;
  //  sel.appendChild(opt);
  Array.from(new Set(metadata.map(r => r.POLUENTE))).sort().forEach(pol=>{{
    const opt = document.createElement('option'); opt.value=pol; opt.textContent=pol;
    sel.appendChild(opt);
  }});
  //if(sel.options.length) sel.selectedIndex = 0;
  //populateStationSelectByPol();
}}

function populateStationSelectByPol() {{
  const pol = (document.getElementById('pollutantSelect').value||'').toString().trim();
  const stSelect = document.getElementById('stationSelect');
  // se o usuário já escolheu via mapa, não sobrescrever (mas se select estiver disabled -> preencher)
  const wasDisabled = stSelect.disabled;
  stSelect.innerHTML = '';
  if(!pol) return;
  const uniqueStations = Array.from(new Set(metadata.filter(r => (r.POLUENTE||'').toString().trim()===pol).map(r => (r.ID_MMA||'').toString().trim()))).filter(Boolean).sort();
  uniqueStations.forEach(st=>{{
    const opt = document.createElement('option'); opt.value = st; opt.textContent = st; stSelect.appendChild(opt);
  }});
  if(wasDisabled) stSelect.disabled = false;
}}

/* ========== Ave-time dropdown (usa CSV de fases) ========== */
function normalizePol(str) {{
  if(!str) return '';
  return str.toString()
            .normalize('NFD')  // separa acentos
            .replace(/[̀-ͯ]/g, '') // remove acentos
            .replace(/\s+/g,'') // remove espaços
            .toLowerCase();
}}
function updateAveTimeDropdown(){{
  const selectedPollutant = normalizePol(document.getElementById('pollutantSelect').value);
  const aveTimeSelect = document.getElementById('aveTimeSelect');
  aveTimeSelect.innerHTML = '';

  const aveTimes = [...new Set(fasesData
    .filter(r => normalizePol(r.pollutant) === selectedPollutant)
    .map(r => (r.ave_time||'').toString().trim())
    .filter(Boolean)
  )].sort();

  if(!aveTimes.length){{
    const opt = document.createElement('option');
    opt.value='';
    opt.textContent='-- none --';
    aveTimeSelect.appendChild(opt);
    console.warn('Nenhum tempo de média encontrado para:', selectedPollutant);
    console.log('Poluentes disponíveis no CSV:', [...new Set(fasesData.map(r=>normalizePol(r.pollutant)))]);
    return;
  }}

  aveTimes.forEach(at => {{
    const o = document.createElement('option');
    o.value = at;
    o.textContent = at;
    aveTimeSelect.appendChild(o);
  }});
  aveTimeSelect.selectedIndex = 0;

  console.log('Dropdown atualizado com tempos de média:', aveTimes);
}}


/* ========== Carrega dados da combinação estação+poluente (forma correta) ========== */
function loadStationData(){{
  const pol = (document.getElementById('pollutantSelect').value || '').toString().trim();
  const st = (document.getElementById('stationSelect').value || '').toString().trim();
  const aveTimeSelect = (document.getElementById('aveTimeSelect').value || '').toString().trim(); 
    
  if(!pol || !st){{ alert('Clique em uma estação no mapa (ou escolha na lista) e selecione um poluente.'); return; }}

  // encontra a linha do metadata que corresponde à combinação (usa ID_MMA_COMPLETO para montar o arquivo)
  const row = metadata.find(r => (r.ID_MMA||'').toString().trim() === st && (r.POLUENTE||'').toString().trim() === pol);
  if(!row){{ alert('Combinação estação+poluente não encontrada no metadata.'); return; }}

  const fileId = (row.ID_MMA_COMPLETO || row.ID_MMA || '').toString().trim();
  if(!fileId){{ alert('ID_MMA_COMPLETO ausente no metadata para esta combinação.'); return; }}

  // monta URL: pasta do poluente / arquivo ID_MMA_COMPLETO.csv  (igual ao primeiro código)
  const fileUrl = `${{DATA_FOLDER}}/${{encodeURIComponent(aveTimeSelect)}}/${{encodeURIComponent(pol)}}/${{encodeURIComponent(fileId)}}.csv`;
  // console.log('Carregando', fileUrl);

  Papa.parse(fileUrl, {{
    download: true,
    header: true,
    skipEmptyLines: true,
    complete: function(res){{
      currentData = res.data || [];
      if(!currentData.length){{ alert('Arquivo carregado, mas sem linhas. Verifique o CSV.'); document.getElementById('plotBtn').disabled = true; return; }}
      alert(`Dados carregados: ${{currentData.length}} linhas.`);
      document.getElementById('plotBtn').disabled = false;
      document.getElementById('downloadBtn').disabled = true;
      setDateLimits(currentData);
    }},
    error: function(err){{
      console.error('Erro ao carregar arquivo:', err);
      alert(`Erro ao carregar arquivo: ${{fileUrl}}\nVerifique console.`)
      document.getElementById('plotBtn').disabled = true;
      document.getElementById('downloadBtn').disabled = true;
    }}
  }});
}}

/* Define limites de datas nos inputs a partir dos dados (usa colunas ANO/MES/DIA/HORA quando presente) */
function setDateLimits(data){{
  if(!data || !data.length) return;
  let dates = data.map(r=>{{
    // tenta extrair ANO MES DIA HORA (formato do dataset original)
    const y = parseInt((r.ANO||r.Year||'').toString(),10);
    const m = parseInt((r.MES||r.Month||'').toString(),10);
    const d = parseInt((r.DIA||r.Day||'').toString(),10);
    const h = parseInt((r.HORA||r.Hour||'0').toString(),10) || 0;
    if([y,m,d].some(v=>Number.isNaN(v))) return null;
    return new Date(y, m-1, d, h);
  }}).filter(d=>d instanceof Date && !isNaN(d));
  if(!dates.length) {{
    // tenta colunas com timestamp/Date
    const alt = data.map(r => {{
      const keys = Object.keys(r);
      for(const k of keys){{
        const v = r[k];
        if(typeof v === 'string' && /^\d{{4}}-\d{{2}}-\d{{2}}/.test(v)) return new Date(v);
      }}
      return null;
    }}).filter(d=>d instanceof Date && !isNaN(d));
    dates = alt;
  }}
  if(!dates.length) return;
  const min = new Date(Math.min(...dates)), max = new Date(Math.max(...dates));
  const pad = n => String(n).padStart(2,'0');
  const startStr = `${{min.getFullYear()}}-${{pad(min.getMonth()+1)}}-${{pad(min.getDate())}}T${{pad(min.getHours())}}:${{pad(min.getMinutes())}}`;
  const endStr   = `${{max.getFullYear()}}-${{pad(max.getMonth()+1)}}-${{pad(max.getDate())}}T${{pad(max.getHours())}}:${{pad(max.getMinutes())}}`;
  document.getElementById('startDate').value = startStr;
  document.getElementById('endDate').value = endStr;
  document.getElementById('startDate').min = startStr;
  document.getElementById('startDate').max = endStr;
  document.getElementById('endDate').min = startStr;
  document.getElementById('endDate').max = endStr;
}}

/* ========== Plot (complexo, com médias, fases, shapes) ========== */
function doPlot(){{
  if(!currentData.length){{ alert('Nenhum dado carregado.'); return; }}

  // Converte CSV bruto em array {{datetime, valor}} e aplica filtro de intervalo
  const startVal = document.getElementById('startDate').value;
  const endVal = document.getElementById('endDate').value;
  const start = startVal ? new Date(startVal) : null;
  const end = endVal ? new Date(endVal) : null;

  // map raw rows -> datetime and numeric value (assume colunas ANO/MES/DIA/HORA e VALOR)
  filteredData = currentData.map(r=>{{
    const y = parseInt((r.ANO||'').toString(),10);
    const m = parseInt((r.MES||'').toString(),10);
    const d = parseInt((r.DIA||'').toString(),10);
    const h = parseInt((r.HORA||'0').toString(),10) || 0;
    const dt = (Number.isNaN(y) || Number.isNaN(m) || Number.isNaN(d)) ? null : new Date(y, m-1, d, h);
    let val = null;
    if(r.VALOR !== undefined && r.VALOR !== null){{
      const num = parseFloat(String(r.VALOR).replace(',', '.').trim());
      if(!Number.isNaN(num) && num >= 0) val = num;
    }}
    return {{ datetime: dt, valor: val }};
  }}).filter(r => r.datetime && (!start || r.datetime >= start) && (!end || r.datetime <= end))
    .sort((a,b) => a.datetime - b.datetime);

  if(!filteredData.length){{ alert('Sem dados no intervalo selecionado.'); return; }}

  const pollutant = (document.getElementById('pollutantSelect').value || '').toString().trim();
  const ave_time = (document.getElementById('aveTimeSelect').value || '').toString().trim().toLowerCase();
  const station = (document.getElementById('stationSelect').value||'').toString().trim();

  // --- APPLY TIME AVERAGING (24h, 8h rolling, annual) ---
  let plottedData = filteredData.slice(); // cópia para manipular


  // numeric array for stats
  const numericVals = plottedData.map(p => p.valor).filter(v => v != null && !isNaN(v));
  const maxVal = numericVals.length ? Math.max(...numericVals)*1.1 : 10;

  // --- FILTER & CONSTRUCT PHASE SHAPES from fasesData ---
  const fasesForPol = fasesData.filter(r => ((r.pollutant||'').toString().trim()).startsWith(pollutant) && (!ave_time || (r.ave_time||'').toString().trim().toLowerCase() === ave_time));
  const plotShapes = fasesForPol.map(r=>{{
    let y0 = parseFloat(String(r.y0||r.y0_val||'').replace(',','.'));
    let y1 = parseFloat(String(r.y1||r.y1_val||'').replace(',','.'));
    if (Number.isNaN(y0)) y0 = null;
    if (Number.isNaN(y1)) y1 = null;

    // If PI-1 and not annual apply "to top" logic (as in your prior code)
    if (!(ave_time.includes("year") || ave_time.includes("ano") || ave_time.includes("anual") || ave_time.includes("annual"))){{
      if ((r.phase||'').toString().trim() === 'PI-1' && y1 !== null) {{
        // set y1 to maxVal so PI-1 extends upward
        y1 = maxVal;
      }}
    }}

    if(y0 === null && y1 === null) return null;

    // if y1 missing, extend to maxVal
    const yLow = (y0 === null) ? 0 : y0;
    const yHigh = (y1 === null) ? maxVal : y1;

    return {{
      type:'rect',
      xref:'paper',
      x0:0, x1:1,
      y0: yLow,
      y1: yHigh,
      fillcolor: r.color || 'rgba(200,200,200,0.25)',
      line: {{ width: 0 }},
      layer: 'below',
      name: r.phase || ''
    }};
  }}).filter(Boolean);

  // shape legends as traces (small trick to show legend)
  const shapeLegends = plotShapes.map(s => ({{
    x:[null], y:[null], mode:'lines', line:{{color:s.fillcolor, width:10}}, name:s.name
  }}));

  // --- BUILT PLOT DATA ---
  let plotData = [];
  if (ave_time.includes("year") || ave_time.includes("ano") || ave_time.includes("anual") || ave_time.includes("annual")) {{
    // bar plot for annual values
    // const years = plottedData.map(p => p.datetime);
    // const values = plottedData.map(p => p.valor);
    const agregados = plottedData.reduce((acc, item) => {{
        const ano = new Date(item.datetime).getFullYear();
        if (!acc[ano]) {{
            acc[ano] = {{ soma: 0, qtd: 0 }};
        }}
        acc[ano].soma += item.valor;
        acc[ano].qtd++;
        return acc;
        }}, {{}});

    // agora criamos arrays para plotar:
    const years = Object.keys(agregados).map(Number); // [2021, 2022, ...]
    const values = Object.values(agregados).map(d => d.soma / d.qtd); // médias por ano

    plotData.push({{
      type:'bar',
      x: years,
      y: values,
      marker: {{ opacity: 0.6 }},
      name: 'Média anual'
    }});
    var layout = {{
      xaxis: {{ title: 'Ano', tickformat: '%Y', dtick: "M12" }},
      yaxis: {{ title: 'Concentração', range: [0, Math.max(...values)*1.1], rangemode: 'nonnegative' }},
      shapes: plotShapes,
      title: `${{(station||'')}} — ${{(pollutant||'')}} —  Série anual `,
      hovermode: 'x unified',
      plot_bgcolor:'rgba(0.95,0.95,0.95,0.4)',
      showlegend: true
    }};
  }} else {{
    // timeseries
    plotData.push({{
      type: 'scatter',
      mode: 'lines',
      x: plottedData.map(p => p.datetime),
      y: plottedData.map(p => p.valor),
      line: {{ color: 'blue', width: 1.5 }},
      connectgaps: false,
      name: pollutant || 'Série'
    }});
    var layout = {{
      title: `${{(station||'')}} — ${{(pollutant||'')}} — ${{(ave_time||'')}}h`,
      hovermode: 'x unified',
      plot_bgcolor: 'rgba(0.95,0.95,0.95,0.4)',
      yaxis: {{ title: 'Concentração', range: [0, maxVal], rangemode: 'nonnegative' }},
      xaxis: {{ title: 'Data/Hora' }},
      shapes: plotShapes,
      showlegend: true
    }};
  }}

  plotData = plotData.concat(shapeLegends);
  Plotly.newPlot('plotDiv', plotData, layout, {{responsive:true}});

  // habilita download se houver valores
  document.getElementById('downloadBtn').disabled = plottedData.length === 0;

  // tabela de ultrapassagensências (usa fasesForPol e numericVals)
  buildExceedanceTable(fasesForPol, numericVals);
}}

/* ========== Tabela de ultrapassagensência ========== */
function buildExceedanceTable(fasesForPol, numericVals){{
  const container = document.getElementById('statsTable');
  container.innerHTML = '';
  if(!numericVals.length || !fasesForPol.length){{
    container.innerHTML = "<p><i>Sem dados ou sem padrões para calcular ultrapassagens.</i></p>";
    return;
  }}

  let html = '<table class="stats-table">';
  html += "<thead><tr><th>Padrão / Fase</th><th>Limite<br>(µg/m³)</th><th>Excedências</th><th>%<br>Excedência</th></tr></thead><tbody>";
  const n = numericVals.length;

  fasesForPol.forEach(r => {{
    const y1 = parseFloat(String(r.y1 || '').replace(',', '.'));
    const totalCount = numericVals.length;
    const validVals = numericVals.filter(v => v != null && !isNaN(v));
    const missingVals = totalCount - validVals.length;
    const missingPerc = ((missingVals / totalCount) * 100).toFixed(1);

    if (Number.isNaN(y1)) return;

    const exceedCount = validVals.filter(v => v > y1).length;
    const perc = ((exceedCount / totalCount) * 100).toFixed(1);

    html += `<tr style="background:${{(r.color || 'transparent')}}20">
      <td>${{r.phase || '-'}}</td>
      <td>${{y1}}</td>
      <td>${{exceedCount}}</td>
      <td>${{perc}}%</td>
    </tr>`;
  }});

  html += "</tbody></table>";
  container.innerHTML = html;
}}

/* ========== Download CSV ========== */
function downloadFilteredCSV(){{
  // Reconstituir os dados exibidos no gráfico (re-parsing a partir de currentData com os mesmos passos do doPlot)
  if(!currentData.length){{ alert('Nenhum dado carregado para download.'); return; }}
  // Gera csv com colunas DataHora (ISO) e Valor com o mesmo processo de filtragem que doPlot aplicou
  // Para simplicidade: usa a variável plotted data obtida de doPlot via leitura do gráfico
  const gd = document.getElementById('plotDiv');
  if(!gd.data || !gd.data.length){{ alert('Nenhum dado plotado para download.'); return; }}

  // tenta extrair trace principal (tipo scatter ou bar)
  const trace = gd.data.find(t => t.name && t.type === 'scatter') || gd.data[0];
  if(!trace){{ alert('Não foi possível extrair dados para download.'); return; }}

  // para scatter: trace.x (Date) e trace.y (values)
  const rows = [];
  if(trace.x && trace.y && trace.x.length === trace.y.length){{
    for(let i=0;i<trace.x.length;i++){{
      const dt = (trace.x[i] instanceof Date) ? trace.x[i].toISOString() : new Date(trace.x[i]).toISOString();
      const val = trace.y[i];
      rows.push({{ DataHora: dt, Valor: val }});
    }}
  }} else {{
    alert('Formato de dados inesperado para exportação.');
    return;
  }}

  const csvString = Papa.unparse(rows);
  const blob = new Blob([csvString], {{ type: 'text/csv;charset=utf-8;' }});
  const url = URL.createObjectURL(blob);
  const pollutant = (document.getElementById('pollutantSelect').value||'').toString().trim();
  const station = (document.getElementById('stationSelect').value||'').toString().trim();
  const filename = `${{station}}_${{pollutant}}_filtrado.csv`;
  const a = document.createElement('a'); a.href = url; a.download = filename; document.body.appendChild(a); a.click(); document.body.removeChild(a);
  URL.revokeObjectURL(url);
}}

/** EVENTS **/
document.addEventListener('DOMContentLoaded', function() {{
  const stationSel = document.getElementById('stationSelect');
  stationSel.addEventListener('change', function() {{
    const selectedOption = stationSel.options[stationSel.selectedIndex];
    console.log(selectedOption.value);
    selectStation(selectedOption.value);
  }});
  
}});
</script>
"""
    return html_code


def render_html_map(html_code, height=1100):
    encoded = base64.b64encode(html_code.encode("utf-8")).decode("utf-8")
    data_uri = f"data:text/html;charset=utf-8;base64,{encoded}"
    return IFrame(src=data_uri, width="100%", height=height)

In [3]:
html_code = build_stations_explorer(UF)
render_html_map(html_code)

In [4]:
# Salva o HTML em um arquivo local e abre no navegador
import os
import webbrowser

output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.abspath(os.path.join(output_dir, "Figura.37.html"))

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_code)

webbrowser.open(f"file://{output_path}")
print(f"Figura 37 salva em: {output_path}")

Figura 37 salva em: /home/nobre/Notebooks/Guia_RQAr/Estadual/secao_4/outputs/figura37.html


### Aviso: 

Não obtivemos o registro histório de todas as estações de monitoramento instaladas no Brasil. 
